<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Training_ALIGNN_model_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Table of contents

1. Installing [ALIGNN](https://github.com/usnistgov/alignn)
2. Example training for regression on 50 materials,
3. Using more than 70 pre-trained models to make fast predictions
4. ALIGNN-FF Ase Calculator: predict energy/forces, optimized strcture, EV curves and phonons
5. Train ALIGNN-FF on a new dataset
6. Additional examples and dataset preparation example
7. Add ALIGNN-results to the JARVIS-Leaderboard

Author: Kamal Choudhary (kamal.choudhary@nist.gov)

## ALIGNN model ref.: https://www.nature.com/articles/s41524-021-00650-1
## ALIGNN-FF ref.: https://pubs.rsc.org/en/content/articlehtml/2023/dd/d2dd00096b

# 1. Installation

In [ ]:
!nvcc --version

Bases on nvcc version, corresponding gpu version of dgl will be needed.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!conda --version

In [ ]:
import condacolab
condacolab.check()

Takes about 6 minutes to install packages

In [ ]:
%%time
!conda install alignn dgl=2.1.0 pytorch torchvision torchaudio pytorch-cuda -c pytorch -c nvidia -y --quiet

In [ ]:
# %%time
# !pip install -q dgl==1.0.1+cu117 -f https://data.dgl.ai/wheels/cu117/repo.html
# !pip install -q alignn phonopy

In [ ]:
# %%time
# !pip install --pre dgl -f https://data.dgl.ai/wheels/cu118/repo.html
# !pip install --pre dglgo -f https://data.dgl.ai/wheels-test/repo.html
# !pip install alignn phonopy

In [ ]:
import os
!pwd
os.chdir('/content')
# Clone ALIGNN repo to get example folder
if not os.path.exists('alignn'):
  !git clone https://github.com/usnistgov/alignn.git

os.chdir('alignn')
# Install using setup.py in case pip didn't work
# !python setup.py develop

#!pip install dgl-cu111 # Colab has cuda 11.1

Example folder with id_prop.csv and 'POSCAR files.'

In [ ]:
!ls "alignn/examples/sample_data"

# 2. Example regression task

# 50 materials and their bandgap data generated with the script [generate_sample_data_reg.py](https://github.com/usnistgov/alignn/blob/main/alignn/examples/sample_data/scripts/generate_sample_data_reg.py)

# Train a model for 3 epochs and batch size of 2. Other parameters are provided in `config_example.json` file.

#Note: For an involved training, use higher batch size such as 16 and epochs such as 300.

Command line train_folder.py is used below.

In [ ]:
import time
t1=time.time()
!train_alignn.py --root_dir "alignn/examples/sample_data" --epochs 3 --batch_size 2 --config "alignn/examples/sample_data/config_example.json" --output_dir=temp
t2=time.time()
print ('Time in s',t2-t1)

In [ ]:
!ls

The model produces *.pt files which are the trained models.

In [ ]:
!ls temp

We can load a trained model above as the following:

In [ ]:
from alignn.models.alignn_atomwise import ALIGNNAtomWise, ALIGNNAtomWiseConfig
import torch
output_features =  1
filename = 'temp/best_model.pt'
device = "cpu"
if torch.cuda.is_available():
    device = torch.device("cuda")
model = ALIGNNAtomWise(ALIGNNAtomWiseConfig(name="alignn_atomwise", output_features=output_features))
model.load_state_dict(torch.load(filename, map_location=device))
model.eval()

Now, we can build graph for a given structure and make a prediction as follows:

In [ ]:
from jarvis.core.atoms import Atoms
from alignn.graphs import Graph
cutoff = 8.0
model=model.to(device)
max_neighbors = 12
atoms = Atoms.from_poscar('alignn/examples/sample_data/POSCAR-JVASP-10.vasp')
g, lg = Graph.atom_dgl_multigraph(
    atoms, cutoff=float(cutoff), max_neighbors=max_neighbors,
)
out_data = (
    model([g.to(device), lg.to(device)])['out']
    .detach()
    .cpu()
    .numpy()
    .flatten()
    .tolist()
)
print ('output', out_data[0])

We have already trained multiple models on several large datasets which can be used with the pretrained.py executable.

# 3. Using more than 70 pre-trained models to make fast predictions

Use pretrained models such as models trained on JARVIS-DFT, QM9, Materials project, hMOF etc. databases. The models are downloaded from figshare. See the list here: https://github.com/usnistgov/alignn/blob/main/alignn/pretrained.py#L28

In [ ]:
!pretrained.py -h

In [ ]:
!pretrained.py --model_name jv_formation_energy_peratom_alignn --file_format poscar --file_path alignn/examples/sample_data/POSCAR-JVASP-10.vasp

#  4. ALIGNN-FF Ase Calculator: predict energy/forces, optimized strcture, EV curves and phonons


ASE Calculator wrapper for ALIGNN-FF

In [ ]:
!pip install -q phonopy

In [ ]:
%%time
from alignn.ff.ff import phonons,ForceField
from jarvis.core.atoms import ase_to_atoms
from jarvis.io.vasp.inputs import Poscar
from alignn.ff.ff import AlignnAtomwiseCalculator,default_path,wt10_path,fd_path
model_path = wt10_path() #default_path()
calc = AlignnAtomwiseCalculator(path=model_path)

# Copper POSCAR file, feel free to try another structure in POSCAR format
cu="""Cu
1.0
1.7985 1.7985 0.0
0.0 1.7985 1.7985
1.7985 0.0 1.7985
Cu
1
Cartesian
0.0 0.0 0.0
"""

pos = Poscar.from_string(cu)
atoms = pos.atoms
ase_atoms = atoms.ase_converter()
model_path = wt10_path() #default_path()
ff = ForceField(
    jarvis_atoms=atoms,
    model_path=model_path,
    stress_wt=0.3,
    force_multiplier=1,
    force_mult_natoms=False,
)
opt, en, fs = ff.optimize_atoms()  # logfile=None)
# There are more than one ALIGNN-FF models such as wt10_path, fd_path etc., check them before reporting
# See: https://github.com/usnistgov/alignn/blob/develop/alignn/ff/ff.py#L67
ph_path=wt10_path()  #fd_path()
ph=phonons(model_path=ph_path,atoms=opt,phonopy_bands_figname="phonopy_bands_cu.png")
thermal = ph.run_thermal_properties()
thermal_prop = ph.get_thermal_properties_dict()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.axis('off')
plt.imshow(plt.imread("phonopy_bands_cu.png"))
plt.show()

In [ ]:
# X-ray diffraction
from jarvis.analysis.diffraction.xrd import XRD
two_thetas, d_hkls, intensities = XRD().simulate(atoms=atoms)
plt.bar(two_thetas,intensities)
plt.xlabel('2$\theta$')
plt.ylabel('Intensity')
plt.show()

In [ ]:
thermal_prop.keys()

In [ ]:
from ase import Atom, Atoms
import numpy as np
import matplotlib.pyplot as plt
model_path = wt10_path() #default_path()
calc = AlignnAtomwiseCalculator(path=model_path)

lattice_params = np.linspace(3.5, 3.8)
fcc_energies = []
ready = True
for a in lattice_params:
    atoms = Atoms([Atom('Cu', (0, 0, 0))],
                  cell=0.5 * a * np.array([[1.0, 1.0, 0.0],
                                           [0.0, 1.0, 1.0],
                                           [1.0, 0.0, 1.0]]),
                 pbc=True)

    atoms.set_tags(np.ones(len(atoms)))

    atoms.calc = calc

    e = atoms.get_potential_energy()
    fcc_energies.append(e)

Check if EV curves are smooth

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.plot(lattice_params, fcc_energies)
plt.title('1x1x1')
plt.xlabel('Lattice constant ($\AA$)')
plt.ylabel('Total energy (eV)')
plt.show()

In [ ]:
from ase import data
for metal in ['Cu', 'Ag', 'Au', 'Ni', 'Pd', 'Pt', 'Rh', 'Ir']:
    a = data.reference_states[data.atomic_numbers[metal]]['a']
    LC = np.linspace(0.9 * a, 1.1 * a)
    vols, energies = [], []
    for a in LC:
        atoms = Atoms([Atom(metal, (0, 0, 0))],
                      cell=0.5 * a * np.array([[1.0, 1.0, 0.0],
                                               [0.0, 1.0, 1.0],
                                               [1.0, 0.0, 1.0]]),
                     pbc=True)

        atoms.set_tags(np.ones(len(atoms)))

        atoms.calc = calc

        e = atoms.get_potential_energy()
        vols += [atoms.get_volume()]
        energies.append(e)
    plt.figure()
    plt.plot(vols, energies)
    plt.title(metal)
    plt.xlabel('volume ($\AA^3$)')
    plt.ylabel('Total energy (eV)');

Using ALIGNN-FF pretrained model to get unrelaxed energy and relaxed structure

Check phonons

In [ ]:
%%time
from alignn.ff.ff import phonons
import numpy as np
from alignn.ff.ff import AlignnAtomwiseCalculator,default_path,wt10_path,alignnff_fmult,fd_path,ForceField
from jarvis.core.atoms import ase_to_atoms
from ase import Atom, Atoms as AseAtoms
a = 3.597
atoms = AseAtoms([Atom('Cu', (0, 0, 0))],
              cell=0.5 * a * np.array([[1.0, 1.0, 0.0],
                                        [0.0, 1.0, 1.0],
                                        [1.0, 0.0, 1.0]]),
              pbc=True)
ph_path=fd_path()
ph=phonons(model_path=ph_path, phonopy_bands_figname="phonopy_bands.png", atoms=ase_to_atoms(atoms))
%matplotlib inline
plt.axis('off')
plt.imshow(plt.imread("phonopy_bands.png"))
plt.show()

In [ ]:
from jarvis.core.atoms import ase_to_atoms
ase_to_atoms(atoms)

In [ ]:
!run_alignn_ff.py --file_path alignn/examples/sample_data/POSCAR-JVASP-10.vasp --task="unrelaxed_energy"

In [ ]:
!run_alignn_ff.py --file_path alignn/examples/sample_data/POSCAR-JVASP-10.vasp --task="optimize"

In [ ]:
!run_alignn_ff.py --file_path alignn/examples/sample_data/POSCAR-JVASP-10.vasp --task="ev_curve"

# 5. Train ALIGNN-FF on a new dataset

For a detailed example, check outthis notebook: https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Train_ALIGNNFF_Mlearn.ipynb

Convert calculations such as from vasprun.xml to id_prop.json format for ALIGNN-FF training, e.g. https://colab.research.google.com/gist/knc6/5513b21f5fd83a7943509ffdf5c3608b/make_id_prop.ipynb

Note: make add_reverse_forces True and lg_on_fly True

In [ ]:
!train_alignn.py --root_dir "alignn/examples/sample_data_ff" --config "alignn/examples/sample_data_ff/config_example_atomwise.json" --output_dir=temp

The generated model is saved as best_model.pt

In [ ]:
!ls -altr temp

# 6. Additional examples:Training [JARVIS-DFT](https://jarvis.nist.gov/jarvisdft) 2D exfoliation energy model, training [QM9](http://quantum-machine.org/datasets/) U0 model

#Train a model for JARVIS-DFT 2D Exfoliation energy

There are quite a few datasets available here:https://jarvis-tools.readthedocs.io/en/master/databases.html
In the following example, we will use the JARVIS-DFT 2D dataset

Get data in id_prop.csv format

In [ ]:
from jarvis.db.figshare import data as jdata
from jarvis.core.atoms import Atoms
import os

cwd = os.getcwd() #current working directory
temp_dir_name = "DataDir_ExfoEnergy"
os.makedirs(temp_dir_name)
os.chdir(temp_dir_name)

dft_3d = jdata("dft_3d")
prop = "exfoliation_energy" #"optb88vdw_bandgap"
f = open("id_prop.csv", "w")
# count = 0
for i in dft_3d:
    atoms = Atoms.from_dict(i["atoms"])
    jid = i["jid"]
    poscar_name = "POSCAR-" + jid + ".vasp"
    target = i[prop]
    if target != "na":
        atoms.write_poscar(poscar_name)
        f.write("%s,%6f\n" % (poscar_name, target))
        # count += 1
        # if count == max_samples:
        #     break
f.close()

os.chdir(cwd)

In [ ]:
!ls -altr  DataDir_ExfoEnergy/*.vasp | wc -l


In [ ]:
 !wc -l DataDir_ExfoEnergy/id_prop.csv

In [ ]:
import time
t1=time.time()
!train_alignn.py --root_dir "DataDir_ExfoEnergy" --epochs 1 --batch_size 64 --config "alignn/examples/sample_data/config_example.json" --output_dir="ExfoEnOut"
t2=time.time()
print ('Time in s',t2-t1)

In [ ]:
!ls ExfoEnOut

Here checkpoints are the model parameter files that can be loaded in torch library to make predictions such as [this example](https://github.com/usnistgov/alignn/blob/main/alignn/scripts/predict.py).

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
df = pd.read_csv('/content/alignn/ExfoEnOut/prediction_results_test_set.csv')

These are predictions on 10 % held dataset that the model has never seen

In [ ]:
df

In [ ]:
# !pip install matplotlib==3.1.3

Obviously, this is just an example. For production runs, please select larger dataset, batch size, epochs etc.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.plot(df['target'],df['prediction'],'.')
plt.plot(df['target'],df['target'],'-.')
plt.xlabel('DFT data (meV)')
plt.ylabel('ALIGNN predictions (meV)')

In [ ]:
# from jarvis.db.figshare import data as jdata
# from jarvis.core.atoms import Atoms
# import os

# temp_dir_name = "DataDir_QM9_U0"
# os.makedirs(temp_dir_name)
# os.chdir(temp_dir_name)

# dft_3d = jdata("qm9_std_jctc") #jdata("dft_3d")
# prop = "U0" #"exfoliation_energy" #"optb88vdw_bandgap"
# f = open("id_prop.csv", "w")
# # count = 0
# for i in dft_3d:
#     atoms = Atoms.from_dict(i["atoms"])
#     jid = i["id"]
#     poscar_name = "POSCAR-" + jid + ".vasp"
#     target = i[prop]
#     if target != "na":
#         atoms.write_poscar(poscar_name)
#         f.write("%s,%6f\n" % (poscar_name, target))
#         # count += 1
#         # if count == max_samples:
#         #     break
# f.close()

# os.chdir(cwd)

In [ ]:
pwd

In [ ]:
!ls

Changing the cut-off in config_example.json as 5,  n_train as 110000, n_val as 10000, n_test as 10829

In [ ]:
# from jarvis.db.jsonutils import loadjson, dumpjson
# config = loadjson("alignn/examples/sample_data/config_example.json")
# config['cutoff'] = 5.0
# config['n_train'] = 110000
# config['n_val'] = 10000
# config['n_test'] = 10829
# dumpjson(data=config, filename="config_qm9.json")

Run for 1000 epochs instead of 5 here to get reasonable performance/MAE

In [ ]:
# import time
# t1=time.time()
# !train_folder.py --root_dir "DataDir_QM9_U0" --epochs 5 --batch_size 64 --config "config_qm9.json" --output_dir="DataDir_QM9U0out"
# t2=time.time()
# print ('Time in s',t2-t1)

# Generate an id_prop.json from vasprun.xml files

In [ ]:
import requests,zipfile,io,tempfile,os
from jarvis.io.vasp.outputs import Vasprun
from jarvis.db.figshare import get_jid_data
from jarvis.db.jsonutils import dumpjson, loadjson
# We will download vasprun.xml from JARVIS-DFT elastic tensor calculations for Silicon (JVASP-1002)
jid = 'JVASP-1002'
x=get_jid_data(jid=jid,dataset='dft_3d')
for i in x['raw_files']:
  if 'FD-ELAST' in i:
    print(i)
    tmp = i.split(".zip,")[1]
    r = requests.get(tmp)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    vrun_path = z.read("vasprun.xml").decode("utf-8")
    fd, path = tempfile.mkstemp()
    with os.fdopen(fd, "w") as tmp:
        tmp.write(vrun_path)
    vrun_elast = Vasprun(path)
for i in x['raw_files']:
  if 'DFT-SCF' in i:
    print(i)
    tmp = i.split(".zip,")[1]
    r = requests.get(tmp)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    vrun_path = z.read("vasprun.xml").decode("utf-8")
    fd, path = tempfile.mkstemp()
    with os.fdopen(fd, "w") as tmp:
        tmp.write(vrun_path)
    vrun_scf = Vasprun(path)

In [ ]:
x['raw_files']

In [ ]:
jid='JVASP-1002'
forces = vrun_elast.all_forces
energies = vrun_elast.all_energies
stresses = vrun_elast.all_stresses
structures = vrun_elast.all_structures
count = 0
mem=[]
for i, j, k, s in zip(energies, forces, stresses, structures):
    count += 1
    jid_c = jid + "_elastic_" + str(count)
    info = {}
    info["jid"] = jid_c
    info["energy"] = i / s.num_atoms
    info["forces"] = j.tolist()
    info["stresses"] = k.tolist()
    info["atoms"] = s.to_dict()
    mem.append(info)



In [ ]:
len(mem)

In [ ]:

forces = vrun_scf.all_forces
energies = vrun_scf.all_energies
stresses = vrun_scf.all_stresses
structures = vrun_scf.all_structures
count = 0
mem=[]
for i, j, k, s in zip(energies, forces, stresses, structures):
    count += 1
    jid_c = jid + "_scf_" + str(count)
    info = {}
    info["jid"] = jid_c
    info["energy"] = i / s.num_atoms
    info["forces"] = j.tolist()
    info["stresses"] = k.tolist()
    info["atoms"] = s.to_dict()
    mem.append(info)



In [ ]:
len(mem)

In [ ]:
dumpjson(data=mem, filename="id_prop.json")

# 7: Add results to JARVIS-Leaderboard


Checkout the notebook: https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/alignn_jarvis_leaderboard.ipynb

In [ ]:
!pip freeze